# Kubernetes with kubeadm {background-color="black" background-image="https://images.unsplash.com/photo-1667372459567-3853510dd5ce?q=80&w=2532&auto=format&fit=crop&ixlib=rb-4.1.0&ixid=M3wxMjA3fDB8MHxwaG90by1wYWdlfHx8fGVufDB8fHx8fA%3D%3D" background-size="cover" background-opacity="0.5"}


# Why Kubernetes? {background-color="#326CE5"}


## From Docker Swarm to Kubernetes

::::{.columns}

::: {.fragment .column width="30%"}

**Docker Swarm**

- A single `docker stack deploy` command deploys a whole app
- Manages a cluster with Manager and Workers
- Great for scaling a single multi-container app on familiar Docker syntax

::: {.callout-note  .fragment}
But what happens when you have more applications, or more complex requirements?
:::

:::

::: {.fragment .column width="70%"}

**Kubernetes**

| Feature | Swarm | Kubernetes |
|---|---|---|
| Workload types | Service | Deployment, StatefulSet, DaemonSet, Job… |
| Scheduling | Basic | Resource requests, affinity, taints |
| Storage | Named volumes | PBC , CSI |
| Networking | Overlay | CNI plugin ecosystem |
| Self-healing | ✅ | ✅ + liveness/readiness probes |
| Ecosystem | Small | Helm, Operators, Service Mesh… |
| Cloud managed | ❌ (*) | EKS, GKE, AKS |


:::{.callout-note .fragment}
(*) AWS ECS / Azure Container Instances/Apps have a similar Swarm-like approach but they are indipendent services, not Kubernetes distributions.
:::
:::

::::


# Kubernetes Architecture {background-color="white" background-opacity="0.7"}
![Components of Kubernetes](https://kubernetes.io/images/docs/components-of-kubernetes.svg)

## kubeadm: The Bootstrap Tool
<https://kubernetes.io/docs/reference/setup-tools/kubeadm/>

::::{.columns}

::: {.fragment .column width="50%"}

> *"kubeadm performs the actions necessary to get a minimum viable cluster up and running"*
> [kubernetes.io](https://kubernetes.io/docs/reference/setup-tools/kubeadm/)

- Runs **pre-flight checks** (swap off, ≥ 2 CPUs, open ports) [Details](https://kubernetes.io/docs/reference/setup-tools/kubeadm/implementation-details/)
- Generates TLS certificates for all components
- Writes static Pod manifests → `kube-apiserver`, `etcd`, `scheduler`
- Bootstraps a join token for worker nodes

::: {.callout-note .fragment title="KaaS"}
Managed services (EKS, GKE, AKS) run kubeadm under the hood
:::

:::

::: {.fragment .column width="50%"}
Logical steps to set up a cluster with kubeadm:

1. On the control-plane node ```kubeadm init --pod-network-cidr=10.244.0.0/16 --control-plane-endpoint=<IP>```
2. For example, to install [Flannel](https://github.com/flannel-io/flannel): ```kubectl apply -f https://github.com/flannel-io/flannel/releases/latest/download/kube-flannel.yml```
3. For each worker node, kubeadm will output a `kubeadm join` to be run on the worker to join the cluster. It looks like this: ```kubeadm join <IP>:6443 --token <token>```

:::{.callout-note .fragment title="CNI"}
A CNI (Container Network Interface) is required for pod networking, kubeadm does not install one by default, since it depends on the use case and platform.
:::
:::
::::


# Lab Setup {background-color="#2C3E50"}


## Cluster Topology

::::{.columns}

::: {.fragment .column width="50%"}

**Lab directory: `lab/k8s-multipass/`**

```
lab/k8s-multipass/
├── terraform/
│   ├── main.tf          # Multipass VMs + inventory
│   ├── variables.tf     # Counts, CPU, RAM, disk
│   ├── outputs.tf       # IPs + next-steps hint
│   ├── cloud-init.tpl   # SSH key injection
│   ├── inventory.tpl    # Ansible inventory template
│   └── id_ed25519(.pub) # SSH key pair
└── ansible/
    ├── 00-prerequisites.yml
    ├── 01-control-plane.yml
    └── 02-workers.yml
```

:::

::: {.fragment .column width="50%"}

**Default sizing (kubeadm minimum)**

| Node | Count | CPUs | RAM | Disk |
|------|-------|------|-----|------|
| control-plane | 1 | 2 | 2 GB | 10 GB |
| worker | 2 | 2 | 2 GB | 10 GB |

::: {.callout-warning}
`kubeadm` **refuses to initialise** if a node has fewer than 2 CPUs.
The default Multipass sizing (1 CPU / 1 GB) must be overridden in `variables.tf`.
:::

:::

::::


## Provision VMs with OpenTofu

::::{.columns}

::: {.fragment .column width="50%"}

**Step 0 — Generate SSH key pair**

```bash
cd lab/k8s-multipass/terraform
ssh-keygen -t ed25519 -C "k8s-lab" -N "" -f id_ed25519
```

**Step 1 — Init and apply**

```bash
tofu init
tofu plan
tofu apply -auto-approve
```

Expected:
```
multipass_instance.control_plane[0]: Creating...
multipass_instance.worker[0]: Creating...
multipass_instance.worker[1]: Creating...

Apply complete! Resources: 5 added.
```

:::

::: {.fragment .column width="50%"}

**Verify VMs are running**

```bash
multipass list
# Name              State   IPv4
# control-plane-1   Running 192.168.64.10
# worker-1          Running 192.168.64.11
# worker-2          Running 192.168.64.12
```

**Ansible inventory (auto-generated)**

```ini
[control_plane]
control-plane-1 ansible_host=192.168.64.10 ansible_user=ubuntu

[workers]
worker-1 ansible_host=192.168.64.11 ansible_user=ubuntu
worker-2 ansible_host=192.168.64.12 ansible_user=ubuntu

[k8s:children]
control_plane
workers
```

:::

::::


# Node Prerequisites {background-color="#7D3C98"}


## Why Every Node Needs Preparation

::::{.columns}

::: {.fragment .column width="50%"}

**kubeadm pre-flight checks — will fail without:**

| Requirement | Why |
|---|---|
| Swap disabled | kubelet cannot manage memory with swap on |
| ≥ 2 CPUs | control-plane components are CPU-hungry |
| `overlay` module | containerd needs it for layered filesystems |
| `br_netfilter` module | iptables must see bridged traffic |
| IP forwarding on | Pods must route packets between nodes |
| `SystemdCgroup = true` | kubelet + containerd must share the same cgroup driver |

:::

::: {.fragment .column width="50%"}

**Playbook pipeline**

```{mermaid}
flowchart TD
    A["00-prerequisites.yml\n(hosts: k8s — all nodes)"]:::play
    B["01-control-plane.yml\n(hosts: control_plane)"]:::play
    C["02-workers.yml\n(hosts: workers)"]:::play
    A --> B --> C

    click A "" "Swap off, kernel modules, sysctl, containerd, kubeadm packages"
    click B "" "kubeadm init, Flannel CNI, capture join command"
    click C "" "kubeadm join on each worker, wait for Ready"

    classDef play fill:#7D3C98,color:#fff,stroke:#6c3483
```

::: {.callout-note}
Run the playbooks **in order** — workers cannot join before the control plane is initialised.
:::

:::

::::


## Ansible: 00-prerequisites.yml

::::{.columns}

::: {.fragment .column width="50%"}

**Targets: `hosts: k8s`** (all nodes)

```yaml
- name: Disable swap immediately
  command: swapoff -a

- name: Load kernel modules (overlay, br_netfilter)
  community.general.modprobe:
    name: "{{ item }}"
  loop: [overlay, br_netfilter]

- name: Set sysctl for Kubernetes networking
  ansible.posix.sysctl:
    name: "{{ item.key }}"
    value: "{{ item.value }}"
  loop:
    - { key: net.bridge.bridge-nf-call-iptables, value: "1" }
    - { key: net.ipv4.ip_forward,                value: "1" }
```

:::

::: {.fragment .column width="50%"}

```yaml
- name: Install containerd
  apt:
    name: containerd

- name: Patch SystemdCgroup = true
  shell: |
    containerd config default \
      | sed 's/SystemdCgroup = false/SystemdCgroup = true/' \
      > /etc/containerd/config.toml

- name: Install kubelet, kubeadm, kubectl (pkgs.k8s.io v1.31)
  apt:
    name: [kubelet, kubeadm, kubectl]

- name: Hold packages to prevent accidental upgrades
  dpkg_selections:
    name: "{{ item }}"
    selection: hold
  loop: [kubelet, kubeadm, kubectl]
```

::: {.callout-tip}
`apt-mark hold` prevents `apt upgrade` from inadvertently breaking the cluster.
:::

:::

::::


# Control-Plane Bootstrap {background-color="#1B4F72"}


## kubeadm init — What Happens

::::{.columns}

::: {.fragment .column width="50%"}

```{mermaid}
flowchart TD
    PF["🔍 Pre-flight checks\n(swap, CPUs, ports, modules)"]:::step
    CERT["🔐 Generate TLS certificates\n(CA, apiserver, etcd, kubelet)"]:::step
    STATIC["📄 Write static Pod manifests\n(apiserver, etcd, scheduler, controller-mgr)"]:::step
    BOOT["🪙 Bootstrap token\nfor worker join"]:::step
    CNI["🌐 Wait for CNI\n(you must apply it!)"]:::step
    PF --> CERT --> STATIC --> BOOT --> CNI

    click PF "https://kubernetes.io/docs/reference/setup-tools/kubeadm/kubeadm-init/#before-you-begin" "Checks swap, kernel modules, available ports and container runtime"
    click CERT "https://kubernetes.io/docs/setup/best-practices/certificates/" "Self-signed CA + leaf certs for every component"
    click CNI "https://kubernetes.io/docs/concepts/extend-kubernetes/compute-storage-net/network-plugins/" "Without CNI all Pods stay Pending — we use Flannel"

    classDef step fill:#1B4F72,color:#fff,stroke:#154360
```

:::

::: {.fragment .column width="50%"}

**After init — mandatory steps**

```bash
# 1. Set up kubeconfig for ubuntu user
mkdir -p ~/.kube
cp /etc/kubernetes/admin.conf ~/.kube/config

# 2. Install Flannel CNI
#    (without this, ALL pods stay Pending)
kubectl apply -f \
  https://github.com/flannel-io/flannel/releases/\
latest/download/kube-flannel.yml

# 3. Verify control-plane is Ready
kubectl get nodes
# NAME              STATUS   ROLES           AGE
# control-plane-1   Ready    control-plane   2m
```

::: {.callout-important}
Until Flannel is installed, even system pods in `kube-system` stay `Pending`. This is expected.
:::

:::

::::


## Ansible: 01-control-plane.yml

::::{.columns}

::: {.fragment .column width="50%"}

**Targets: `hosts: control_plane`**

```yaml
- name: Run kubeadm init
  command: >
    kubeadm init
    --pod-network-cidr=10.244.0.0/16
    --control-plane-endpoint={{ ansible_host }}
  when: not kubeadm_init_done.stat.exists

- name: Copy admin.conf → ~/.kube/config
  copy:
    src: /etc/kubernetes/admin.conf
    dest: /home/ubuntu/.kube/config
    owner: ubuntu
```

:::

::: {.fragment .column width="50%"}

```yaml
- name: Install Flannel CNI
  command: kubectl apply -f {{ flannel_manifest }}
  environment:
    KUBECONFIG: /home/ubuntu/.kube/config

- name: Capture join command
  command: kubeadm token create --print-join-command
  register: join_command_raw

- name: Save join command on the controller
  local_action:
    module: copy
    content: "{{ join_command_raw.stdout }}"
    dest: /tmp/k8s_join_command.sh
```

::: {.callout-tip}
`local_action` saves the join command **on the Ansible controller** — the next playbook reads it and distributes it to every worker.
:::

:::

::::


# Worker Nodes {background-color="#1A5276"}


## Ansible: 02-workers.yml — Join & Verify

::::{.columns}

::: {.fragment .column width="50%"}

**Targets: `hosts: workers`**

```yaml
- name: Check if already joined
  stat:
    path: /etc/kubernetes/kubelet.conf
  register: already_joined

- name: Read join command from controller
  set_fact:
    join_command: "{{ lookup('file',
      '/tmp/k8s_join_command.sh') }}"
  when: not already_joined.stat.exists

- name: Join the cluster
  command: "{{ join_command }}"
  when: not already_joined.stat.exists
```

:::

::: {.fragment .column width="50%"}

**Verify from control-plane**

```bash
kubectl get nodes
# NAME              STATUS   ROLES           AGE
# control-plane-1   Ready    control-plane   5m
# worker-1          Ready    <none>          2m
# worker-2          Ready    <none>          2m

kubectl get nodes -o wide
# also shows IPs and container runtime
```

::: {.callout-note}
The playbook **polls every 15 s** (up to 5 min) until all nodes reach `Ready`. Flannel may take a moment to assign Pod CIDRs to newly joined nodes.
:::

:::

::::


# kubectl {background-color="#326CE5"}


## Core Kubernetes Objects

::::{.columns}

::: {.fragment .column width="55%"}

| Object | Description |
|---|---|
| **Pod** | Smallest unit — one or more containers sharing network + storage |
| **Deployment** | Manages a ReplicaSet; rolling updates, rollbacks |
| **Service** | Stable virtual IP + DNS for a set of Pods |
| **Namespace** | Logical isolation boundary |
| **ConfigMap** | Key/value config injected into Pods |
| **Secret** | Like ConfigMap but for sensitive data |
| **PersistentVolumeClaim** | A Pod's request for durable storage |

:::

::: {.fragment .column width="45%"}

**Service types**

| Type | Exposes to |
|---|---|
| `ClusterIP` | Inside the cluster only |
| `NodePort` | Host port on every node |
| `LoadBalancer` | External LB (cloud) |
| `ExternalName` | DNS alias |

::: {.callout-tip}
For this lab we use **NodePort** — it opens a port (30000–32767) on every node's IP, directly reachable from the host machine.
:::

:::

::::


## Essential kubectl Commands

::::{.columns}

::: {.fragment .column width="50%"}

**Inspect**

```bash
kubectl get nodes                     # cluster nodes
kubectl get pods -A                   # all pods, all namespaces
kubectl get pods -o wide              # with node + IP
kubectl describe pod <name>           # full event log + conditions
kubectl logs <pod>                    # stdout / stderr
kubectl exec -it <pod> -- bash        # shell into pod
```

**Apply / delete**

```bash
kubectl apply -f manifest.yaml        # declarative apply
kubectl delete -f manifest.yaml       # remove by manifest
kubectl delete pod <name>             # remove by name
```

:::

::: {.fragment .column width="50%"}

**Deployments**

```bash
# Create imperatively
kubectl create deployment nginx \
  --image=nginx --replicas=3

# Expose as NodePort
kubectl expose deployment nginx \
  --type=NodePort --port=80

# Scale
kubectl scale deployment nginx --replicas=6

# Rolling update
kubectl set image deployment/nginx nginx=nginx:alpine

# Check rollout progress
kubectl rollout status deployment/nginx

# Rollback
kubectl rollout undo deployment/nginx
```

:::

::::


# Hands-on {background-color="#17202A"}


## Deploy & Expose a Workload

::::{.columns}

::: {.fragment .column width="50%"}

**Deploy nginx across workers**

```bash
kubectl create deployment nginx \
  --image=nginx --replicas=3

kubectl expose deployment nginx \
  --type=NodePort --port=80

# Get the assigned NodePort
kubectl get svc nginx
# NAME    TYPE       CLUSTER-IP   PORT(S)
# nginx   NodePort   10.96.x.x    80:3xxxx/TCP
```

:::

::: {.fragment .column width="50%"}

**Access from the host machine**

```bash
WORKER_IP=$(tofu -chdir=lab/k8s-multipass/terraform \
  output -json worker_ips \
  | python3 -c \
  'import json,sys; print(json.load(sys.stdin)[0])')

NODE_PORT=$(kubectl get svc nginx \
  -o jsonpath='{.spec.ports[0].nodePort}')

curl http://${WORKER_IP}:${NODE_PORT}
# <h1>Welcome to nginx!</h1>

# Pods spread across nodes
kubectl get pods -o wide
```

::: {.callout-tip}
Any worker IP + NodePort works — `kube-proxy` forwards the request to a healthy pod regardless of which node receives it.
:::

:::

::::


## Scaling & Rolling Updates

::::{.columns}

::: {.fragment .column width="50%"}

**Scale up**

```bash
kubectl scale deployment nginx --replicas=6
kubectl get pods -o wide
# Pods are spread across both workers
```

**Rolling update — zero downtime**

```bash
kubectl set image deployment/nginx \
  nginx=nginx:alpine
kubectl rollout status deployment/nginx
# Waiting for deployment to finish...
# deployment "nginx" successfully rolled out
```

**Rollback**

```bash
kubectl rollout undo deployment/nginx
kubectl rollout status deployment/nginx
```

:::

::: {.fragment .column width="50%"}

**How rolling updates work**

```{mermaid}
flowchart LR
    OLD["nginx:latest\n6 pods"]:::old
    SURGE["nginx:latest × 5\nnginx:alpine × 1"]:::mid
    MID["nginx:latest × 3\nnginx:alpine × 3"]:::mid
    DONE["nginx:alpine\n6 pods"]:::new

    OLD -->|"set image"| SURGE --> MID --> DONE

    classDef old  fill:#E74C3C,color:#fff,stroke:#a93226
    classDef mid  fill:#F39C12,color:#fff,stroke:#b7770d
    classDef new  fill:#27AE60,color:#fff,stroke:#1a7a42
```

::: {.callout-tip}
By default: `maxSurge: 1`, `maxUnavailable: 0`.  
The old version **keeps serving traffic** until the new pod passes its readiness probe.
:::

:::

::::


## Full Cluster Lifecycle

```{mermaid}
flowchart LR
    KEY(["🔑 ssh-keygen"]):::tf
    TF(["⚙️ tofu apply"]):::tf
    VMS["Multipass VMs\ncontrol-plane-1\nworker-1, worker-2"]:::vm
    PRE(["📦 00-prerequisites.yml"]):::ans
    CP(["🚀 01-control-plane.yml"]):::ans
    WK(["🔗 02-workers.yml"]):::ans
    READY["☸️ Cluster Ready\n3 × Ready nodes"]:::k8s
    DEPLOY["📋 kubectl apply\nDeployment + Service"]:::k8s
    DESTROY(["💣 tofu destroy"]):::tf

    KEY --> TF --> VMS --> PRE --> CP --> WK --> READY --> DEPLOY --> DESTROY

    click KEY "" "Generate id_ed25519 key pair used by Ansible to SSH into nodes"
    click TF "https://opentofu.org" "Provisions Multipass VMs and generates the Ansible inventory"
    click PRE "https://kubernetes.io/docs/setup/production-environment/tools/kubeadm/install-kubeadm/" "Disables swap, loads kernel modules, installs containerd + kubeadm"
    click CP "https://kubernetes.io/docs/reference/setup-tools/kubeadm/kubeadm-init/" "kubeadm init + Flannel CNI + capture join command"
    click WK "https://kubernetes.io/docs/reference/setup-tools/kubeadm/kubeadm-join/" "kubeadm join on each worker, waits for all nodes to be Ready"
    click DESTROY "https://opentofu.org/docs/cli/commands/destroy/" "Stops and deletes all Multipass VMs"

    classDef tf   fill:#7B68EE,color:#fff,stroke:#4b3fa0
    classDef vm   fill:#27AE60,color:#fff,stroke:#1a7a42
    classDef ans  fill:#E74C3C,color:#fff,stroke:#a93226
    classDef k8s  fill:#326CE5,color:#fff,stroke:#1a4a9f
```


## Cleanup

::::{.columns}

::: {.fragment .column width="50%"}

**Remove workloads**

```bash
kubectl delete deployment nginx
kubectl delete service nginx
```

**Destroy all VMs**

```bash
cd lab/k8s-multipass/terraform
tofu destroy -auto-approve
```

Expected:
```
multipass_instance.worker[0]: Destroying...
multipass_instance.worker[1]: Destroying...
multipass_instance.control_plane[0]: Destroying...

Destroy complete! Resources: 5 destroyed.
```

:::

::: {.fragment .column width="50%"}

**Verify nothing remains**

```bash
multipass list
# No instances found.
```

::: {.callout-important}
Always `tofu destroy` when you're done.  
Each running VM consumes host CPU + RAM.  
Multipass VMs are **not** automatically cleaned up on reboot.
:::

:::

::::


# What's Next? {background-color="#161b22" background-image="https://about.gitea.com/gitea.svg" background-size="10%" background-position="90% 15%"}


## Lesson 10 — GitOps for Kubernetes with Gitea

::::{.columns}

::: {.fragment .column width="50%"}

**What we did manually today**

```bash
# Terminal 1 — infrastructure
tofu apply
ansible-playbook 00-prerequisites.yml
ansible-playbook 01-control-plane.yml
ansible-playbook 02-workers.yml

# Terminal 2 — application
kubectl apply -f k8s/
kubectl rollout status deployment/nginx
```

Same problem as Lessons 3→4: not reproducible, no audit trail, credential on your laptop.

:::

::: {.fragment .column width="50%"}

**Lesson 10: automate everything with Gitea**

```{mermaid}
flowchart TD
    PUSH_INFRA["git push\nterraform/** or ansible/**"]:::git
    PUSH_APP["git push\nk8s/**"]:::git
    INFRA["infra.yml\ntofu apply\n+ 3 Ansible playbooks\n+ save kubeconfig"]:::infra
    DEPLOY["deploy.yml\nkubectl apply -f k8s/"]:::deploy
    CLUSTER["☸️ K8s Cluster\nReady"]:::k8s
    APP["🚀 nginx running\non NodePort"]:::k8s

    PUSH_INFRA --> INFRA --> CLUSTER
    PUSH_APP --> DEPLOY --> APP
    CLUSTER -.->|kubeconfig saved on host| DEPLOY

    click INFRA "" "Shell runner on host — has direct access to multipass, tofu, ansible"
    click DEPLOY "" "KUBECONFIG=~/k8s-config kubectl apply — no Ansible needed for K8s workloads"

    classDef git    fill:#333,color:#fff,stroke:#555
    classDef infra  fill:#7B68EE,color:#fff,stroke:#4b3fa0
    classDef deploy fill:#326CE5,color:#fff,stroke:#1a4a9f
    classDef k8s    fill:#27AE60,color:#fff,stroke:#1a7a42
```

:::

::::

::: {.callout-tip .fragment}
**Key design principle:** Ansible owns the cluster (OS → kubeadm). `kubectl` owns the workloads. Gitea pipelines automate both — with separate triggers.
:::


## Summary

| Step | Tool | Command |
|------|------|---------|
| Generate SSH key | ssh-keygen | `ssh-keygen -t ed25519 -f id_ed25519` |
| Provision VMs | OpenTofu | `tofu apply` |
| Generate inventory | Terraform template | *(automatic)* |
| Install prerequisites | Ansible | `ansible-playbook 00-prerequisites.yml` |
| Init control plane | Ansible + kubeadm | `ansible-playbook 01-control-plane.yml` |
| Join workers | Ansible + kubeadm | `ansible-playbook 02-workers.yml` |
| Verify cluster | kubectl | `kubectl get nodes` |
| Deploy workload | kubectl | `kubectl create deployment …` |
| Expose service | kubectl | `kubectl expose deployment … --type=NodePort` |
| Scale | kubectl | `kubectl scale deployment … --replicas=N` |
| Destroy | OpenTofu | `tofu destroy` |

::: {.callout-tip .fragment}
**Next lesson:** Helm charts, Ingress controllers, persistent storage — or managed Kubernetes on AWS with EKS.
:::
